# Estrategia de ofertas FDS 3-5 julio 2026 - Pricing Justo

Version integrada del equipo: combina el modelo economico (score de priorizacion,
guardrail de utilidad incremental, tope autonomo de descuento, castigo por posicionamiento,
escenarios de sensibilidad - ver `src/Estrategia_Pricing_FDS_Julio.ipynb` para la version
original sin Snowflake) con datos reales de Snowflake en vez de supuestos declarados:
elasticidad y unidades/dia reales de `ATHENEA` (la version original usaba
`ELASTICIDAD_POR_TAG`/`Q0_POR_ROTACION` como supuestos porque no tenia acceso a Snowflake),
Departamento/Categoria real de `FULL_MASTER_CATALOG`, y un backtesting contra promociones
historicas reales que ademas valida el supuesto de multiplicador promocional (`MULT_PROMO`).

**Restriccion dura:** ningun SKU puede quedar con margen menor a 22% despues de la oferta.
**Restriccion de negocio nueva:** ninguna oferta se publica si su utilidad incremental
proyectada es negativa (no tiene sentido descontar algo que ya se vende bien sin ayuda).
**Tope autonomo:** descuentos que requieran mas de 30% de profundidad para tener sentido
economico se marcan para aprobacion de Comercial, no se publican solos.

## Parametros del modelo

Los supuestos declarados por no tener Snowflake en la version original (`ELASTICIDAD_POR_TAG`,
`Q0_POR_ROTACION`) NO estan aqui - se reemplazan mas abajo por datos reales de `ATHENEA`,
con fallback en cascada (SKU real -> promedio de categoria real -> supuesto declarado como
ultimo recurso, documentado donde se usa). El resto de parametros son el modelo economico
original, sin cambios.

In [ ]:
import pandas as pd
import numpy as np
import re

# ---------- Insumos ----------
RUTA_OPORTUNIDAD = "../data/inputs/oportunidad.xlsx"
RUTA_DESCUENTOS  = "../data/inputs/Descuentos comerciales.xlsx"
RUTA_SALIDA      = "../data/output/estrategia_fds.xlsx"

# ---------- Fin de semana objetivo ----------
WEEKEND_INICIO = pd.Timestamp("2026-07-03")   # viernes
WEEKEND_FIN    = pd.Timestamp("2026-07-05")   # domingo

# ---------- Reglas de margen ----------
PISO_MARGEN    = 0.22
TOPE_GUARDRAIL = 0.30   # autonomo; por encima requiere Vo.Bo. de Comercial
DESC_MINIMO    = 0.05   # por debajo de esto no vale la pena publicar la oferta

# ---------- Priorizacion (score -> fraccion del colchon autorizada) ----------
TOPES_SCORE        = [(7, 0.25), (5, 0.20), (3, 0.15), (2, 0.10)]
CASTIGO_MAS_BARATO = 0.05   # se resta al tope si el SKU ya es "Mas barato" en linea
BOOST_MUNDIAL      = 1
TOLERANCIA_CASCADA = 0.02

# ---------- Mecanicas flexibles por ticket ----------
TICKET_ALTO = 150   # BNSDP: ahorro en pesos, ticket alto
TICKET_BAJO = 60    # BNSP: paquete de 3 a precio redondo, alta rotacion

# ---------- Modelo economico ----------
MULT_PROMO = 2.0   # amplificacion promocional declarada; se contrasta contra el
                   # backtesting mas abajo para ver si el supuesto se sostiene
REDENCION = {"multibuy_grande": 0.35,   # 4x3 / 5x4 / 6x5
             "pack_3": 0.40,            # BNSP 3 unidades
             "umbral_2": 0.55}          # SPON / BNSDP 2 unidades
ESCENARIOS = {"Conservador": 0.6, "Base": 1.0, "Optimista": 1.4}

pd.set_option("display.max_colwidth", 60)

## Carga y parsing del universo

In [ ]:
oportunidad = pd.read_excel(RUTA_OPORTUNIDAD, sheet_name="Export")
print(f"Filas originales del export: {len(oportunidad)}")

df = oportunidad[oportunidad["SKU + Nombre"].notna() & oportunidad["COSTO"].notna()].copy()
df["STORE_ID"] = pd.to_numeric(df["STORE_ID"], errors="coerce")
df = df[df["STORE_ID"].notna()].copy()
df["STORE_ID"] = df["STORE_ID"].astype(int)

df[["SKU", "Nombre"]] = df["SKU + Nombre"].str.split(" - ", n=1, expand=True)
df["SKU"] = pd.to_numeric(df["SKU"], errors="coerce")
df = df.dropna(subset=["SKU"]).copy()
df["SKU"] = df["SKU"].astype(int)

print(f"Universo (tienda x SKU): {len(df)}")
df["MOTIVO_SIN_OFERTA"] = ""
df.head()

## Cruce comercial: blacklist y campanias que cruzan el FDS

`Descuentos comerciales.xlsx` trae 7 hojas; 4 son operativas. Se toma la **union** de
`DEscuentos comerciales` e `Historico` (esta filtrada a `Estatus == 'Cargado'`, el resto son
codigos de otro proceso) para el calendario de campanias, y se **restan** los pares
de `Eliminacion de campanias` (bajas confirmadas, ya no compiten). Se marca el motivo en
vez de borrar la fila - el entregable debe mostrar la decision sobre cada SKU del universo,
no solo los que sí llevan oferta.

In [ ]:
def pares_limpios(tabla, cols=("ItemId", "AreaId")):
    t = tabla[list(cols)].apply(pd.to_numeric, errors="coerce").dropna().astype(int)
    return set(map(tuple, t.values))


black_list = pd.read_excel(RUTA_DESCUENTOS, sheet_name="Black list")
pares_blacklist = pares_limpios(black_list)

comerciales = pd.read_excel(RUTA_DESCUENTOS, sheet_name="DEscuentos comerciales")
historico = pd.read_excel(RUTA_DESCUENTOS, sheet_name="Historico")
historico = historico[historico["Estatus"] == "Cargado"]

campanias = pd.concat([comerciales, historico], ignore_index=True)
campanias["ItemId"] = pd.to_numeric(campanias["ItemId"], errors="coerce")
campanias["AreaId"] = pd.to_numeric(campanias["AreaId"], errors="coerce")
campanias = campanias.dropna(subset=["ItemId", "AreaId", "FechaInicio", "FechaFin"]).copy()
campanias["ItemId"] = campanias["ItemId"].astype(int)
campanias["AreaId"] = campanias["AreaId"].astype(int)

cruza_fds = campanias[
    (campanias["FechaInicio"] <= WEEKEND_FIN) & (campanias["FechaFin"] >= WEEKEND_INICIO)
]
pares_campania = set(zip(cruza_fds["ItemId"], cruza_fds["AreaId"]))

eliminadas = pd.read_excel(RUTA_DESCUENTOS, sheet_name="Eliminacion de campañas")
pares_eliminados = pares_limpios(eliminadas)

pares_campania_vigente = pares_campania - pares_eliminados

df_pares = pd.Series(list(zip(df["SKU"], df["STORE_ID"])), index=df.index)
en_blacklist = df_pares.isin(pares_blacklist)
en_campania = df_pares.isin(pares_campania_vigente) & ~en_blacklist

df.loc[en_blacklist, "MOTIVO_SIN_OFERTA"] = "Black list"
df.loc[en_campania, "MOTIVO_SIN_OFERTA"] = "Campania comercial vigente en el FDS"

print(f"Excluidos por Black list: {en_blacklist.sum()}")
print(f"Excluidos por campania comercial vigente (cruza el FDS, no eliminada): {en_campania.sum()}")
print(f"Universo restante sin motivo: {(df['MOTIVO_SIN_OFERTA'] == '').sum()}")

## Catalogo real: Departamento / Categoria (Snowflake)

La version original no tenia esto (sus insumos autorizados eran solo los 2 Excel), por eso
su regla de dia de ejecucion usaba mecanica/rotacion como proxy. Con Departamento real,
"Despensa" se usa directamente para el dia Domingo mas abajo.

In [ ]:
import sys
sys.path.insert(0, ".")
from snowflake_conn import get_connection

conn = get_connection()
cur = conn.cursor()
cur.execute("SELECT CURRENT_USER(), CURRENT_ROLE(), CURRENT_WAREHOUSE(), CURRENT_DATABASE()")
print(cur.fetchone())

In [ ]:
cur.execute('''
    SELECT DISTINCT SKU, DEPARTMENT, CATEGORY
    FROM MX_JUSTO_PROD.SANDBOX.FULL_MASTER_CATALOG
''')
columnas = [c[0] for c in cur.description]
catalogo = pd.DataFrame(cur.fetchall(), columns=columnas)
catalogo = catalogo.rename(columns={"DEPARTMENT": "Departamento", "CATEGORY": "Categoria"})
catalogo["SKU"] = pd.to_numeric(catalogo["SKU"], errors="coerce")
catalogo = catalogo.dropna(subset=["SKU"])
catalogo["SKU"] = catalogo["SKU"].astype(int)
catalogo = catalogo.drop_duplicates(subset="SKU", keep="first")

df = df.merge(catalogo, on="SKU", how="left")
cobertura = df["Departamento"].notna().sum()
print(f"Cobertura Departamento/Categoria: {cobertura}/{len(df)} ({cobertura/len(df)*100:.1f}%)")

## Fuente de verdad del margen

`MARGEN` es la definicion oficial de pure margin: margen sobre el precio neto de impuestos,
$P_{neto} = P/[(1+IVA)(1+IEPS)]$. Se valida contra el calculo directo antes de usarla como
unica fuente de verdad - toda la matematica del piso se hace por algebra sobre
$m$ = `MARGEN`/100, sin reconstruir el margen desde costo e impuestos.

In [ ]:
m = df["MARGEN"] / 100
df["PRECIO_NETO"] = df["PRECIO JUSTO"] / (1 + df["IVA"].fillna(0) / 100) / (1 + df["IEPS"].fillna(0) / 100)

precio_neto_check = df["PRECIO JUSTO"] / (1 + df["IVA"] / 100) / (1 + df["IEPS"] / 100)
margen_check = (precio_neto_check - df["COSTO"]) / precio_neto_check * 100
diff = (margen_check - df["MARGEN"]).abs()
print(f"Diferencia vs columna MARGEN: media={diff.mean():.4f} pts, filas con diff>0.5pts={(diff > 0.5).sum()}/{len(df)}")
print(f"Margen del universo -> mediana: {m.median()*100:.1f}%")

## Elegibilidad

Dos filtros, aplicados solo sobre filas aun sin motivo (precedencia: comercial > margen >
elasticidad): margen base < 22% (no hay colchon), o TAG ELAS INELASTICO/SIN DATOS (descontar
donde la demanda no responde no genera nada). POCO ELASTICO se queda: responde poco, pero
responde, y el score mas abajo dosifica la agresividad.

In [ ]:
sin_motivo = df["MOTIVO_SIN_OFERTA"] == ""

m_bajo = (df["MARGEN"] < PISO_MARGEN * 100) & sin_motivo
df.loc[m_bajo, "MOTIVO_SIN_OFERTA"] = "Margen base < 22%"

elas_out = df["TAG ELAS"].isin(["INELASTICO", "SIN DATOS"]) & sin_motivo & ~m_bajo
df.loc[elas_out, "MOTIVO_SIN_OFERTA"] = "Inelastico / sin datos de elasticidad"

df["ELEGIBLE"] = df["MOTIVO_SIN_OFERTA"] == ""
print(f"Excluidos por margen base < 22%: {m_bajo.sum()}")
print(f"Excluidos por inelastico/sin datos: {elas_out.sum()}")
print(f"Elegibles: {df['ELEGIBLE'].sum()}")

## Tema Mundial

El fin de semana cae en octavos de final del Mundial 2026 con Mexico como anfitrion. No hay
merchandising con licencia (cero matches para "mundial"/"futbol"/"FIFA"/"seleccion"), asi que
se usa un proxy de consumo de "ver el partido": cerveza, refrescos, botanas saladas, totopos,
palomitas, salsas picantes - depurado a mano contra falsos positivos (se descarto
"cacahuate", que en este catalogo solo matcheaba crema de cacahuate/yogurt). Doble uso:
+1 punto de score, y el dia de ejecucion cubre **sabado y domingo** (no solo sabado - hay
partidos en ambos dias de este fin de semana).

In [ ]:
PATRON_MUNDIAL = re.compile(
    r"\b(?:"
    r"cerveza|refresco|"
    r"papas?\s+(?:fritas?|pringles|ruffles)|totopos?|nachos?|chicharr\w*|"
    r"palomitas|botanas?|"
    r"habanero|valentina|cholula|clamato|b[uú]falo"
    r")\b",
    re.IGNORECASE,
)
df["TEMA_MUNDIAL"] = df["Nombre"].str.contains(PATRON_MUNDIAL, na=False)
print(f"SKUs marcados como afines al Mundial: {df['TEMA_MUNDIAL'].sum()}")

## Descuento maximo $d_{max}$ y tope autonomo

$d_{max}$ es el colchon puro de margen del SKU (igual que antes: $d \leq 1-(1-m)/0{,}78$).
`DMAX` es ese colchon **acotado a `TOPE_GUARDRAIL` (30%)** - es el que alimenta el resto del
pipeline (score, cascada, modelo economico). `DMAX_REAL` es el colchon sin acotar: donde
`DMAX_REAL > 30%`, el SKU se marca `REQUIERE_APROBACION` y se reporta ademas la mecanica que
usaria si Comercial aprueba mas descuento - no se descarta, se separa para revision.

In [ ]:
df["DMAX_REAL"] = (1 - (1 - m) / (1 - PISO_MARGEN)).clip(lower=0)
df.loc[~df["ELEGIBLE"], "DMAX_REAL"] = 0.0

df["DMAX"] = df["DMAX_REAL"].clip(upper=TOPE_GUARDRAIL)
df["REQUIERE_APROBACION"] = df["DMAX_REAL"] > TOPE_GUARDRAIL

eleg = df[df["ELEGIBLE"]]
print(f"d_max (autonomo, tope 30%) -> mediana: {eleg['DMAX'].median():.1%}")
print(f"SKUs que requieren aprobacion Comercial (d_max real > 30%): {df['REQUIERE_APROBACION'].sum()}")

## Elasticidad y unidades/dia reales (ATHENEA) - reemplaza los supuestos declarados

Fallback en cascada, documentado por fila en `FUENTE_ELASTICIDAD` / `FUENTE_Q0`:
1. **Real por SKU+Tienda** (Athenea, `ELASTICITY_BY_SKU` / `AVG_UNITS_PER_DAY`).
2. **Promedio real por Categoria** (calculado solo con los SKUs de esta corrida que si
   tienen dato propio en Athenea).
3. **Supuesto declarado por tag** (magnitudes de la version original: -2.2/-1.5/-0.8 por
   elasticidad, 20/6/1.5/0.3 u/dia por rotacion) - solo como ultimo recurso, si ni el SKU ni
   su categoria tienen dato real en absoluto.

In [ ]:
cur.execute('''
    SELECT
        PRODUCT_ID::VARCHAR  AS SKU,
        WAREHOUSE_ID         AS STORE_ID,
        ELASTICITY_BY_SKU,
        AVG_UNITS_PER_DAY
    FROM MX_JUSTO_PROD.DR_GENERAL.ATHENEA
    WHERE WAREHOUSE_ID IN (9, 14)
      AND STATUS_SAP = 'Prendido'
      AND ELASTICITY_BY_SKU IS NOT NULL
''')
columnas = [c[0] for c in cur.description]
athenea = pd.DataFrame(cur.fetchall(), columns=columnas)
athenea["SKU"] = pd.to_numeric(athenea["SKU"], errors="coerce")
athenea = athenea.dropna(subset=["SKU"])
athenea["SKU"] = athenea["SKU"].astype(int)
athenea["STORE_ID"] = athenea["STORE_ID"].astype(int)
athenea = athenea.drop_duplicates(subset=["SKU", "STORE_ID"], keep="first")

df = df.merge(athenea, on=["SKU", "STORE_ID"], how="left")
print(f"SKUs con elasticidad propia (Athenea, SKU+Tienda): {df['ELASTICITY_BY_SKU'].notna().sum()}/{len(df)}")

In [ ]:
ELASTICIDAD_POR_TAG_FALLBACK = {"ALTAMENTE ELASTICO": -2.2, "ELASTICO": -1.5, "POCO ELASTICO": -0.8}
Q0_POR_ROTACION_FALLBACK = {"Alta rotacion": 20.0, "Rotacion normal": 6.0,
                             "Baja rotacion": 1.5, "Sin rotacion": 0.3}

elasticidad_por_categoria = df.dropna(subset=["ELASTICITY_BY_SKU"]).groupby("Categoria")["ELASTICITY_BY_SKU"].mean()
unidades_por_categoria = df.dropna(subset=["AVG_UNITS_PER_DAY"]).groupby("Categoria")["AVG_UNITS_PER_DAY"].mean()


def fuente_y_valor(row, col_real, prom_por_cat, fallback_dict, tag_col):
    if pd.notna(row[col_real]):
        return "SKU real", row[col_real]
    prom_cat = prom_por_cat.get(row["Categoria"], float("nan"))
    if pd.notna(prom_cat):
        return "Categoria real", prom_cat
    return "Supuesto declarado", fallback_dict.get(row[tag_col], float("nan"))


res_elas = df.apply(lambda r: fuente_y_valor(r, "ELASTICITY_BY_SKU", elasticidad_por_categoria,
                                              ELASTICIDAD_POR_TAG_FALLBACK, "TAG ELAS"), axis=1)
df["FUENTE_ELASTICIDAD"] = res_elas.str[0]
df["ELASTICIDAD_FINAL"] = res_elas.str[1].astype(float)

res_q0 = df.apply(lambda r: fuente_y_valor(r, "AVG_UNITS_PER_DAY", unidades_por_categoria,
                                            Q0_POR_ROTACION_FALLBACK, "TAG_ROTACION"), axis=1)
df["FUENTE_Q0"] = res_q0.str[0]
df["Q0_DIA"] = res_q0.str[1].astype(float)

print("Fuente de elasticidad:")
print(df.loc[df["ELEGIBLE"], "FUENTE_ELASTICIDAD"].value_counts())
print()
print("Fuente de Q0 (unidades/dia base):")
print(df.loc[df["ELEGIBLE"], "FUENTE_Q0"].value_counts())

## Score y descuento objetivo

$$SCORE = W_{elas} \times W_{rot} + S_{share} + S_{mundial}$$

La multiplicacion es deliberada: un cero en elasticidad **o** en rotacion anula la
prioridad. $S_{share}$ prioriza los SKUs que pesan en la venta; $S_{mundial}$ empuja los de
temporada. El score se traduce a `TOPE` (fraccion del colchon **autonomo** que se autoriza a
usar), y el castigo de posicionamiento resta un nivel a los SKUs que ya son "Mas barato" en
linea - la percepcion de precio ya esta comprada.

In [ ]:
elas_w = {"ALTAMENTE ELASTICO": 3, "ELASTICO": 2, "POCO ELASTICO": 1, "INELASTICO": 0, "SIN DATOS": 0}
rot_w  = {"Alta rotacion": 3, "Rotacion normal": 2, "Baja rotacion": 1, "Sin rotacion": 0}

df["W_ELAS"] = df["TAG ELAS"].map(elas_w).fillna(0)
df["W_ROT"]  = df["TAG_ROTACION"].map(rot_w).fillna(0)
df["SCORE"]  = (df["W_ELAS"] * df["W_ROT"]
                + (df["TAG_SHARE"] == "ALTO SHARE").astype(int)
                + df["TEMA_MUNDIAL"].astype(int) * BOOST_MUNDIAL)


def tope_por_score(s):
    for smin, tope in TOPES_SCORE:
        if s >= smin:
            return tope
    return 0.0


df["TOPE"] = df["SCORE"].apply(tope_por_score)
mas_barato = df["INDEX PRECIO LINEA"] == "Mas barato"
df.loc[mas_barato, "TOPE"] = (df.loc[mas_barato, "TOPE"] - CASTIGO_MAS_BARATO).clip(lower=0)

df["D_TARGET"] = np.minimum(df["DMAX"], df["TOPE"])
df.loc[~df["ELEGIBLE"], "D_TARGET"] = 0.0
print(f"Filas 'Mas barato' castigadas: {mas_barato.sum()}")
print(f"Elegibles con d_target >= {DESC_MINIMO:.0%}: {(df['D_TARGET'] >= DESC_MINIMO).sum()}")

## Asignacion de mecanica

Cascada de mas agresiva a mas suave. Multibuys con nombre primero (con tolerancia: si
$d_{target}$ quedo a menos de 2 puntos del umbral y el margen alcanza, se sube - "6x5"
comunica mejor que "-15% llevando 2"); luego mecanica flexible por ticket: BNSDP en pesos
redondos para ticket alto, BNSP de paquete redondo para ticket bajo de alta rotacion, SPON
por defecto. **Todas exigen 2 o 3 unidades** - el precio de gondola no cambia, quien compra
una unidad paga completo, y el carrito crece.

In [ ]:
UMBRALES_MULTIBUY = [(0.25, "4x3", 3, 4), (0.20, "5x4", 4, 5), (1/6, "6x5", 5, 6)]


def asignar(r):
    d, dmax, p = r["D_TARGET"], r["DMAX"], r["PRECIO JUSTO"]
    if d < DESC_MINIMO:
        return pd.Series(["Sin oferta", p, 0.0])
    for umbral, nombre, n_pag, n_llev in UMBRALES_MULTIBUY:
        if d >= umbral or ((umbral - d <= TOLERANCIA_CASCADA) and (dmax >= umbral)):
            return pd.Series([nombre, round(p * n_pag / n_llev, 2), round(umbral, 4)])
    d5 = np.floor(d * 20) / 20
    if d5 < DESC_MINIMO:
        return pd.Series(["Sin oferta", p, 0.0])
    if p >= TICKET_ALTO:
        ahorro = np.floor(d5 * p / 5) * 5
        if ahorro < 5:
            return pd.Series(["Sin oferta", p, 0.0])
        return pd.Series([f"BNSDP: 2 uds, ahorra ${ahorro:.0f} c/u", round(p - ahorro, 2), round(ahorro / p, 4)])
    if p <= TICKET_BAJO and r["W_ROT"] == 3:
        pack = np.floor(3 * p * (1 - d5))
        return pd.Series([f"BNSP: 3 x ${pack:.0f}", round(pack / 3, 2), round(1 - pack / (3 * p), 4)])
    return pd.Series([f"SPON: 2 uds, -{d5*100:.0f}%", round(p * (1 - d5), 2), d5])


df[["ESTRATEGIA", "PRECIO_OFERTA", "DESC_EFECTIVO"]] = df.apply(asignar, axis=1)
df["MECANICA"] = df["ESTRATEGIA"].str.split(":").str[0]
print(df["MECANICA"].value_counts().to_string())

## Validacion del piso

Recalcula el margen de las unidades vendidas **con la mecanica activa** usando el descuento
efectivo real (incluye redondeos y tolerancia): $m_d = 1-(1-m)/(1-d)$. Cualquier oferta bajo
22% se degrada con motivo - protege cada unidad vendida en promocion, no un promedio.

In [ ]:
df["MARGEN_OFERTA"] = (1 - (1 - m) / (1 - df["DESC_EFECTIVO"])) * 100

viola = (df["MECANICA"] != "Sin oferta") & (df["MARGEN_OFERTA"] < PISO_MARGEN * 100)
df.loc[viola, ["ESTRATEGIA", "DESC_EFECTIVO"]] = ["Sin oferta", 0.0]
df.loc[viola, "PRECIO_OFERTA"] = df.loc[viola, "PRECIO JUSTO"]
df.loc[viola, "MOTIVO_SIN_OFERTA"] = "Degradada: violaria el piso de margen"
df["MECANICA"] = df["ESTRATEGIA"].str.split(":").str[0]
print(f"Degradadas por violar el piso: {viola.sum()}")

## Modelo economico de la promocion

Tres piezas (formulas del modelo original, alimentadas con `ELASTICIDAD_FINAL`/`Q0_DIA`
reales en vez de supuestos por tag):

1. **La demanda responde a la oferta exhibida completa**: $u = |E| \times MULT \times d$,
   $Q_1 = Q_0(1+u)$.
2. **Pero el descuento solo lo paga quien activa la mecanica** (tasa de redencion $r$):
   ingreso por unidad promedio $\bar P = P(1-rd)$, utilidad por unidad promedio
   $\bar\pi = P_{neto}(m-rd)$.
3. **Totales del dia de oferta**: $GMV_1 = Q_1 P(1-rd)$, $\Pi_1 = Q_1 P_{neto}(m-rd)$, contra
   la base $GMV_0=Q_0P$, $\Pi_0=Q_0P_{neto}m$.

In [ ]:
def redencion(mec):
    if mec in ("4x3", "5x4", "6x5"):
        return REDENCION["multibuy_grande"]
    if mec == "BNSP":
        return REDENCION["pack_3"]
    if mec in ("SPON", "BNSDP"):
        return REDENCION["umbral_2"]
    return 0.0


df["REDENCION"] = df["MECANICA"].apply(redencion)
d_eff = df["DESC_EFECTIVO"]
df["UPLIFT"] = df["ELASTICIDAD_FINAL"].abs() * MULT_PROMO * d_eff
df["Q1_DIA"] = df["Q0_DIA"] * (1 + df["UPLIFT"])

rd = df["REDENCION"] * d_eff
df["GMV_BASE_DIA"]  = (df["Q0_DIA"] * df["PRECIO JUSTO"]).round(2)
df["GMV_PROY_DIA"]  = (df["Q1_DIA"] * df["PRECIO JUSTO"] * (1 - rd)).round(2)
df["UTIL_BASE_DIA"] = (df["Q0_DIA"] * df["PRECIO_NETO"] * m).round(2)
df["UTIL_PROY_DIA"] = (df["Q1_DIA"] * df["PRECIO_NETO"] * (m - rd)).round(2)
df["GMV_INC_DIA"]   = (df["GMV_PROY_DIA"]  - df["GMV_BASE_DIA"]).round(2)
df["UTIL_INC_DIA"]  = (df["UTIL_PROY_DIA"] - df["UTIL_BASE_DIA"]).round(2)

ofer = df[df["MECANICA"] != "Sin oferta"]
print(f"Uplift promedio de unidades (ofertas): {ofer['UPLIFT'].mean()*100:.0f}%")
print(f"Descuento mezclado promedio (r x d): {(ofer['REDENCION']*ofer['DESC_EFECTIVO']).mean()*100:.1f}% "
      f"(vs profundidad exhibida {ofer['DESC_EFECTIVO'].mean()*100:.1f}%)")

## Guardrail economico: solo ofertas con utilidad incremental positiva

El piso del 22% protege el margen **unitario**; este guardrail protege el **resultado
total**. Es la traduccion operativa de "no le des descuento a lo que ya se vende bien sin
ayuda": si la venta extra proyectada no compensa el costo mezclado de la promocion
($u \leq rd/(m-rd)$), la oferta se descarta con motivo.

In [ ]:
destruye = (df["MECANICA"] != "Sin oferta") & (df["UTIL_INC_DIA"] < 0)
df.loc[destruye, ["ESTRATEGIA", "DESC_EFECTIVO"]] = ["Sin oferta", 0.0]
df.loc[destruye, "PRECIO_OFERTA"] = df.loc[destruye, "PRECIO JUSTO"]
df.loc[destruye, "MOTIVO_SIN_OFERTA"] = "Descartada: utilidad incremental proyectada < 0"
df["MECANICA"] = df["ESTRATEGIA"].str.split(":").str[0]

anular = df["MECANICA"] == "Sin oferta"
df.loc[anular, ["UPLIFT", "REDENCION"]] = 0.0
df.loc[anular, "Q1_DIA"] = df.loc[anular, "Q0_DIA"]
df.loc[anular, "GMV_PROY_DIA"]  = df.loc[anular, "GMV_BASE_DIA"]
df.loc[anular, "UTIL_PROY_DIA"] = df.loc[anular, "UTIL_BASE_DIA"]
df.loc[anular, ["GMV_INC_DIA", "UTIL_INC_DIA"]] = 0.0
df["MARGEN_OFERTA"] = ((1 - (1 - m) / (1 - df["DESC_EFECTIVO"])) * 100).round(2)

ofer = df[df["MECANICA"] != "Sin oferta"]
print(f"Descartadas por economia negativa: {destruye.sum()}")
print(f"Ofertas finales: {len(ofer)} de {len(df)} ({len(ofer)/len(df)*100:.1f}%)")
if len(ofer):
    print(f"Utilidad incremental minima por SKU: ${ofer['UTIL_INC_DIA'].min():,.2f}")

## Confianza de la proyeccion y backtesting contra historico real

Complementa el modelo economico con evidencia: cada oferta se etiqueta **Alta/Media/Baja**
segun si su elasticidad es real por SKU y si ese SKU ya tuvo una promocion real en el pasado
con la que comparar. El backtesting va mas alla de la version original: en vez de solo
sensibilizar el multiplicador `MULT_PROMO` con factores arbitrarios (0.6/1.0/1.4), se compara
la proyeccion (con ese mismo `MULT_PROMO`) contra el uplift **real** observado en
`MASTER_ORDERLINE` durante promociones pasadas - permite decir si 2.0 es razonable o hay que
ajustarlo.

In [ ]:
cur.execute('''
    SELECT
        REGEXP_REPLACE(PRODUCT_ID, '[^0-9]', '') AS SKU,
        WAREHOUSE_ID                              AS STORE_ID,
        START_DATE,
        END_DATE,
        BY_BULK_STRATEGY_DISCOUNT_TYPE            AS TIPO_MECANICA,
        BY_BULK_STRATEGY_VALUE                    AS VALOR_DESCUENTO
    FROM MX_JUSTO_PROD.ODS_MS_PRICING_V.DISCOUNT_CAMPAIGN_EVENT
    WHERE START_DATE >= '2025-01-01'
''')
columnas = [c[0] for c in cur.description]
promos_historicas = pd.DataFrame(cur.fetchall(), columns=columnas)
promos_historicas["SKU"] = pd.to_numeric(promos_historicas["SKU"], errors="coerce")
promos_historicas = promos_historicas.dropna(subset=["SKU", "STORE_ID", "START_DATE", "END_DATE"])
promos_historicas["SKU"] = promos_historicas["SKU"].astype(int)
promos_historicas["STORE_ID"] = promos_historicas["STORE_ID"].astype(int)

skus_elegibles = set(df.loc[df["ELEGIBLE"], "SKU"])
promos_relevantes = promos_historicas[promos_historicas["SKU"].isin(skus_elegibles)].copy()
skus_con_historial_promo = set(zip(promos_relevantes["SKU"], promos_relevantes["STORE_ID"]))
print(f"Pares (SKU, Tienda) elegibles con al menos una promo pasada: {len(skus_con_historial_promo)}")

In [ ]:
def nivel_confianza(row):
    if row["FUENTE_ELASTICIDAD"] != "SKU real":
        return "Baja"
    tiene_historial = (row["SKU"], row["STORE_ID"]) in skus_con_historial_promo
    return "Alta" if tiene_historial else "Media"


df["CONFIANZA_PROYECCION"] = df.apply(nivel_confianza, axis=1)
print("Distribucion de confianza (elegibles):")
print(df.loc[df["ELEGIBLE"], "CONFIANZA_PROYECCION"].value_counts())

In [ ]:
skus_param = tuple(int(s) for s in skus_elegibles) if skus_elegibles else (0,)
placeholders = ",".join(["%s"] * len(skus_param))

query_ventas = f'''
    SELECT
        PRODUCT_ID::VARCHAR                                    AS SKU,
        STORE_ID,
        TO_DATE(DATETIME_DELIVERY)                             AS FECHA,
        SUM(QUANTITY_FULFILLED_PZ)                             AS UNIDADES
    FROM MX_JUSTO_PROD.DR_MASTER_TABLES.MASTER_ORDERLINE
    WHERE STATUS_ORDER            = 'delivered'
      AND STORE_ID                IN (9, 14)
      AND QUANTITY_FULFILLED_PZ   > 0
      AND DATETIME_DELIVERY       BETWEEN '2025-01-01' AND '2026-10-22'
      AND PRODUCT_ID::VARCHAR     IN ({placeholders})
    GROUP BY 1, 2, 3
    LIMIT 1000000
'''
cur.execute(query_ventas, skus_param)
columnas = [c[0] for c in cur.description]
ventas_historicas = pd.DataFrame(cur.fetchall(), columns=columnas)
ventas_historicas["SKU"] = pd.to_numeric(ventas_historicas["SKU"], errors="coerce")
ventas_historicas = ventas_historicas.dropna(subset=["SKU"])
ventas_historicas["SKU"] = ventas_historicas["SKU"].astype(int)
ventas_historicas["STORE_ID"] = ventas_historicas["STORE_ID"].astype(int)
ventas_historicas["FECHA"] = pd.to_datetime(ventas_historicas["FECHA"])
print(f"Filas de ventas historicas: {len(ventas_historicas)}")
if len(ventas_historicas) >= 1_000_000:
    print("AVISO: se llego al LIMIT de 1,000,000 filas - historico truncado")

In [ ]:
promos_pct = promos_relevantes[
    (promos_relevantes["TIPO_MECANICA"] == "PERCENTAGE") & promos_relevantes["VALOR_DESCUENTO"].notna()
]
elasticidad_real_dict = df.set_index(["SKU", "STORE_ID"])["ELASTICIDAD_FINAL"].to_dict()
fuente_dict = df.set_index(["SKU", "STORE_ID"])["FUENTE_ELASTICIDAD"].to_dict()

filas_backtest = []
for _, promo in promos_pct.iterrows():
    sku, store = promo["SKU"], promo["STORE_ID"]
    if fuente_dict.get((sku, store)) != "SKU real":
        continue
    elasticidad = elasticidad_real_dict.get((sku, store))
    if elasticidad is None or pd.isna(elasticidad):
        continue

    ventas_sku = ventas_historicas[(ventas_historicas["SKU"] == sku) & (ventas_historicas["STORE_ID"] == store)]
    if ventas_sku.empty:
        continue
    en_promo = ventas_sku[(ventas_sku["FECHA"] >= promo["START_DATE"]) & (ventas_sku["FECHA"] <= promo["END_DATE"])]
    fuera_promo = ventas_sku[(ventas_sku["FECHA"] < promo["START_DATE"]) | (ventas_sku["FECHA"] > promo["END_DATE"])]
    if len(en_promo) < 2 or len(fuera_promo) < 5:
        continue
    unidades_promo = en_promo["UNIDADES"].mean()
    unidades_base = fuera_promo["UNIDADES"].mean()
    if unidades_base <= 0:
        continue

    d_hist = float(promo["VALOR_DESCUENTO"]) / 100
    uplift_real = (unidades_promo - unidades_base) / unidades_base * 100
    uplift_proyectado = abs(elasticidad) * MULT_PROMO * d_hist * 100

    filas_backtest.append({
        "SKU": sku, "STORE_ID": store,
        "Uplift real %": round(uplift_real, 1),
        "Uplift proyectado %": round(uplift_proyectado, 1),
        "Error (proyectado - real)": round(uplift_proyectado - uplift_real, 1),
    })

backtest = pd.DataFrame(filas_backtest)
if len(backtest):
    mae = backtest["Error (proyectado - real)"].abs().mean()
    bias = backtest["Error (proyectado - real)"].mean()
    direccion = "sobreestima" if bias > 0 else "subestima"
    print(f"n={len(backtest)} | MAE={mae:.1f}pts | sesgo={bias:+.1f}pts")
    print(f"CONCLUSION: con MULT_PROMO={MULT_PROMO}, el modelo {direccion} el uplift real en "
          f"{abs(bias):.1f}% en promedio. "
          + ("El supuesto parece razonable." if abs(bias) < 10 else
             f"Considerar ajustar MULT_PROMO (sugerido: {MULT_PROMO * (1 - bias/100):.2f})."))
else:
    mae = bias = None
    print("No hay suficientes pares SKU+Tienda con historial de promo % y ventas antes/durante para backtesting")

## Dia de ejecucion

Precedencia (la primera que aplica gana):

1. **Tema Mundial -> Sabado y Domingo**: hay partidos en ambos dias de este fin de semana, la
   oferta corre los dos dias en vez de concentrarse en uno solo.
2. **Departamento Despensa -> Domingo**: reabastecimiento antes de empezar la semana (dato
   real de catalogo, no disponible en la version original).
3. **Alta rotacion -> Viernes**: arranque del fin de semana con los productos de mayor
   trafico.
4. **BNSDP (ticket alto) -> Domingo**: la compra de reposicion/ticket grande se decide con
   mas calma.
5. **Resto -> Sabado**: dia insignia del fin de semana.

In [ ]:
def dia_ejecucion(r):
    if r["MECANICA"] == "Sin oferta":
        return ""
    if r["TEMA_MUNDIAL"]:
        return "Sabado y Domingo"
    if r["Departamento"] == "Despensa":
        return "Domingo"
    if r["TAG_ROTACION"] == "Alta rotacion":
        return "Viernes"
    if r["MECANICA"] == "BNSDP":
        return "Domingo"
    return "Sabado"


df["DIA_EJECUCION"] = df.apply(dia_ejecucion, axis=1)
print(df.loc[df["MECANICA"] != "Sin oferta", "DIA_EJECUCION"].value_counts().to_string())

## Escenarios de sensibilidad

El multiplicador promocional es el supuesto con mas varianza real (ahora contrastado contra
el backtesting de arriba), asi que se somete a sensibilidad: el uplift de cada oferta se
escala 0.6x / 1.0x / 1.4x y se recalculan los totales. Si incluso el escenario conservador
deja utilidad incremental positiva, la estrategia es robusta al supuesto mas fragil del
modelo.

In [ ]:
filas = []
o = df[df["MECANICA"] != "Sin oferta"]
rd_o = o["REDENCION"] * o["DESC_EFECTIVO"]
for nombre, f in ESCENARIOS.items():
    q1 = o["Q0_DIA"] * (1 + o["UPLIFT"] * f)
    gmv1  = (q1 * o["PRECIO JUSTO"] * (1 - rd_o)).sum()
    util1 = (q1 * o["PRECIO_NETO"] * (o["MARGEN"]/100 - rd_o)).sum()
    filas.append({"Escenario": nombre, "Factor uplift": f,
                  "Unidades": round(q1.sum()), "GMV dia oferta": round(gmv1),
                  "GMV incremental": round(gmv1 - o["GMV_BASE_DIA"].sum()),
                  "Utilidad incremental": round(util1 - o["UTIL_BASE_DIA"].sum())})
escenarios_df = pd.DataFrame(filas)
print(f"Base de comparacion (mismas ofertas sin promo): unidades {o['Q0_DIA'].sum():,.0f} | "
      f"GMV ${o['GMV_BASE_DIA'].sum():,.0f} | utilidad ${o['UTIL_BASE_DIA'].sum():,.0f}")
escenarios_df

## Resultados finales

In [ ]:
ofer = df[df["MECANICA"] != "Sin oferta"]

print("=== ESTRATEGIA ===")
print(df["MECANICA"].value_counts().to_string())
print(f"\nOfertas: {len(ofer)} de {len(df)} ({len(ofer)/len(df)*100:.1f}%) | "
      f"descuento exhibido promedio: {ofer['DESC_EFECTIVO'].mean()*100:.1f}% | "
      f"margen minimo en promocion: {ofer['MARGEN_OFERTA'].min():.2f}%")
print(f"SKUs que requieren aprobacion Comercial (no incluidos, d_max real > 30%): "
      f"{df['REQUIERE_APROBACION'].sum()}")

print("\n=== PROYECCION ESCENARIO BASE ===")
print(f"Unidades: {ofer['Q0_DIA'].sum():,.0f} -> {ofer['Q1_DIA'].sum():,.0f} "
      f"({(ofer['Q1_DIA'].sum()/ofer['Q0_DIA'].sum()-1)*100:+.1f}%)")
print(f"GMV:      ${ofer['GMV_BASE_DIA'].sum():,.0f} -> ${ofer['GMV_PROY_DIA'].sum():,.0f} "
      f"({(ofer['GMV_PROY_DIA'].sum()/ofer['GMV_BASE_DIA'].sum()-1)*100:+.1f}%)")
print(f"Utilidad: ${ofer['UTIL_BASE_DIA'].sum():,.0f} -> ${ofer['UTIL_PROY_DIA'].sum():,.0f} "
      f"({(ofer['UTIL_PROY_DIA'].sum()/ofer['UTIL_BASE_DIA'].sum()-1)*100:+.1f}%)")

print("\n=== POR DIA ===")
print(ofer.groupby("DIA_EJECUCION").agg(Ofertas=("SKU", "count"),
      GMV_inc=("GMV_INC_DIA", "sum"), Util_inc=("UTIL_INC_DIA", "sum")).round(0).to_string())

print("\n=== CONFIANZA DE LA PROYECCION (ofertas finales) ===")
print(ofer["CONFIANZA_PROYECCION"].value_counts())

mun = ofer[ofer["TEMA_MUNDIAL"]]
print(f"\n=== MUNDIAL ===\nOfertas: {len(mun)} | GMV incremental: ${mun['GMV_INC_DIA'].sum():,.0f} | "
      f"Utilidad incremental: ${mun['UTIL_INC_DIA'].sum():,.0f}")

if len(backtest):
    print(f"\n=== BACKTESTING ===\nn={len(backtest)} | MAE={mae:.1f}pts | sesgo={bias:+.1f}pts")
else:
    print("\n=== BACKTESTING ===\nSin datos suficientes")

## Exportacion del Excel final

Seis hojas: `Estrategia FDS` (solo las filas con oferta - verde, o dorado si es Mundial;
azul si ademas requiere aprobacion Comercial para ir mas profundo), `Descartados` (el resto
del universo, con su motivo - para poder auditar por que un SKU se quedo fuera), `Resumen`
(indicadores globales, por dia y por mecanica), `Escenarios` (sensibilidad), `Backtesting`
(evidencia real vs proyectado) y `Leyenda` (diccionario de columnas).

In [ ]:
from openpyxl.styles import Font, PatternFill, Alignment
from openpyxl.utils import get_column_letter

cols_out = ["STORE_ID", "SKU", "Nombre", "Departamento", "Categoria", "COSTO", "IEPS", "IVA",
            "MARGEN", "PRECIO JUSTO", "TAG_MARGEN", "INDEX PRECIO DCTO", "PRECIO COMP DCTO",
            "PRECIO COMP LINEA", "INDEX PRECIO LINEA", "TAG_ROTACION", "TAG_SHARE", "TAG ELAS",
            "PROMO ACTIVA HOY", "TEMA_MUNDIAL", "MOTIVO_SIN_OFERTA", "REQUIERE_APROBACION",
            "DMAX_REAL", "ESTRATEGIA", "DIA_EJECUCION", "PRECIO_OFERTA", "DESC_EFECTIVO",
            "MARGEN_OFERTA", "FUENTE_ELASTICIDAD", "FUENTE_Q0", "CONFIANZA_PROYECCION",
            "REDENCION", "ELASTICIDAD_FINAL", "UPLIFT", "Q0_DIA", "Q1_DIA",
            "GMV_BASE_DIA", "GMV_PROY_DIA", "GMV_INC_DIA", "UTIL_BASE_DIA", "UTIL_PROY_DIA", "UTIL_INC_DIA"]
out = df[cols_out].copy()
out["DESC_EFECTIVO"] = (out["DESC_EFECTIVO"] * 100).round(1)
out["DMAX_REAL"] = (out["DMAX_REAL"] * 100).round(1)
out["UPLIFT"] = (out["UPLIFT"] * 100).round(0)
out = out.rename(columns={"DESC_EFECTIVO": "DESC_EFECTIVO_%", "MARGEN_OFERTA": "MARGEN_OFERTA_%",
                          "UPLIFT": "UPLIFT_%", "DMAX_REAL": "DMAX_REAL_%"})
out = out.sort_values(["DIA_EJECUCION", "STORE_ID", "ESTRATEGIA"]).reset_index(drop=True)

out_oferta = out[out["ESTRATEGIA"] != "Sin oferta"].reset_index(drop=True)
out_descartados = out[out["ESTRATEGIA"] == "Sin oferta"].reset_index(drop=True)

resumen = pd.DataFrame({
    "Indicador": ["Fin de semana objetivo", "Universo (tienda x SKU)", "Ofertas asignadas", "% cobertura",
                  "Descuento exhibido promedio", "Margen minimo en promocion",
                  "Requieren aprobacion Comercial (d_max > 30%)", "Ofertas Tema Mundial",
                  "Unidades base -> proyectadas (escenario base)",
                  "GMV base -> proyectado (escenario base)", "Utilidad base -> proyectada (escenario base)",
                  "Utilidad incremental total (base)", "Backtesting (n, MAE, sesgo)", "Guardrails aplicados"],
    "Valor": [f"{WEEKEND_INICIO.date()} a {WEEKEND_FIN.date()}", len(df), len(ofer),
              f"{len(ofer)/len(df)*100:.1f}%", f"{ofer['DESC_EFECTIVO'].mean()*100:.1f}%",
              f"{ofer['MARGEN_OFERTA'].min():.2f}%", int(df["REQUIERE_APROBACION"].sum()),
              int(ofer["TEMA_MUNDIAL"].sum()),
              f"{ofer['Q0_DIA'].sum():,.0f} -> {ofer['Q1_DIA'].sum():,.0f}",
              f"${ofer['GMV_BASE_DIA'].sum():,.0f} -> ${ofer['GMV_PROY_DIA'].sum():,.0f}",
              f"${ofer['UTIL_BASE_DIA'].sum():,.0f} -> ${ofer['UTIL_PROY_DIA'].sum():,.0f}",
              f"${ofer['UTIL_INC_DIA'].sum():,.0f}",
              f"n={len(backtest)}, MAE={mae:.1f}pts, sesgo={bias:+.1f}pts" if len(backtest) else "sin datos",
              "Piso 22% por unidad + utilidad incremental >= 0 + tope autonomo 30%"]
})
por_dia = ofer.groupby("DIA_EJECUCION").agg(Ofertas=("SKU", "count"),
          GMV_incremental=("GMV_INC_DIA", "sum"), Utilidad_incremental=("UTIL_INC_DIA", "sum")).round(0).reset_index()
mecs = df["MECANICA"].value_counts().rename_axis("Mecanica").reset_index(name="SKU_tienda")

leyenda = pd.DataFrame([
    ("MOTIVO_SIN_OFERTA", "Por que la fila no lleva oferta (solo aplica en la hoja Descartados)"),
    ("REQUIERE_APROBACION / DMAX_REAL_%", "True si el colchon de margen real supera el tope autonomo de 30%; DMAX_REAL_% es ese colchon completo"),
    ("TEMA_MUNDIAL", "True si el nombre coincide con el patron de consumo 'ver el partido'"),
    ("ESTRATEGIA / DIA_EJECUCION", "Mecanica asignada y dia(s) del FDS en que corre"),
    ("PRECIO_OFERTA", "Precio por unidad cuando la mecanica se activa"),
    ("DESC_EFECTIVO_% / MARGEN_OFERTA_%", "Profundidad exhibida y pure margin de las unidades en promocion"),
    ("FUENTE_ELASTICIDAD / FUENTE_Q0", "SKU real / Categoria real / Supuesto declarado - de donde salio cada dato"),
    ("CONFIANZA_PROYECCION", "Alta: elasticidad propia + historial de promo real. Media: elasticidad propia, sin historial. Baja: fallback"),
    ("REDENCION", "Fraccion supuesta de unidades que se venden con la mecanica activa"),
    ("ELASTICIDAD_FINAL / UPLIFT_%", "Elasticidad usada (real o fallback) y crecimiento proyectado de unidades"),
    ("Q0_DIA / Q1_DIA", "Unidades/dia sin y con promocion"),
    ("GMV_*_DIA", "Venta en pesos del dia de oferta: base, proyectada, incremental"),
    ("UTIL_*_DIA", "Pure margin en pesos del dia de oferta: base, proyectada, incremental"),
], columns=["Columna", "Descripcion"])

with pd.ExcelWriter(RUTA_SALIDA, engine="openpyxl") as w:
    out_oferta.to_excel(w, sheet_name="Estrategia FDS", index=False)
    out_descartados.to_excel(w, sheet_name="Descartados", index=False)
    resumen.to_excel(w, sheet_name="Resumen", index=False, startrow=0)
    por_dia.to_excel(w, sheet_name="Resumen", index=False, startrow=len(resumen) + 3)
    mecs.to_excel(w, sheet_name="Resumen", index=False, startrow=len(resumen) + 3 + len(por_dia) + 3)
    escenarios_df.to_excel(w, sheet_name="Escenarios", index=False)
    backtest.to_excel(w, sheet_name="Backtesting", index=False)
    leyenda.to_excel(w, sheet_name="Leyenda", index=False)

    wb = w.book
    hdr = PatternFill("solid", start_color="1F4E79")
    verde, dorado, azul = (PatternFill("solid", start_color="E2EFDA"),
                            PatternFill("solid", start_color="FFF2CC"),
                            PatternFill("solid", start_color="DDEBF7"))

    for hoja_nombre in ["Estrategia FDS", "Descartados"]:
        ws = wb[hoja_nombre]
        for c in ws[1]:
            c.font = Font(bold=True, color="FFFFFF", name="Arial", size=10)
            c.fill = hdr
            c.alignment = Alignment(horizontal="center", vertical="center")
        ws.freeze_panes = "D2"
        ws.auto_filter.ref = ws.dimensions
        headers = [c.value for c in ws[1]]
        i_mun = headers.index("TEMA_MUNDIAL")
        i_apr = headers.index("REQUIERE_APROBACION")
        for row in ws.iter_rows(min_row=2):
            if hoja_nombre == "Estrategia FDS":
                f = azul if row[i_apr].value else (dorado if row[i_mun].value else verde)
                for cell in row:
                    cell.fill = f
        for i in range(1, ws.max_column + 1):
            ws.column_dimensions[get_column_letter(i)].width = {3: 50}.get(i, 14)

    for hoja in ["Resumen", "Escenarios", "Backtesting", "Leyenda"]:
        ws2 = wb[hoja]
        for c in ws2[1]:
            c.font = Font(bold=True, color="FFFFFF", name="Arial", size=10)
            c.fill = hdr
        ws2.column_dimensions["A"].width = 46
        ws2.column_dimensions["B"].width = 56

print(f"Exportado: {RUTA_SALIDA} | {len(out_oferta)} en Estrategia FDS | {len(out_descartados)} en Descartados "
      f"({int(ofer['TEMA_MUNDIAL'].sum())} Mundial, {int(df['REQUIERE_APROBACION'].sum())} pendientes de aprobacion)")